In [9]:
is_colab = 'google.colab' in str(get_ipython())
is_colab


True

## Installing and importing nessisary dependencies

In [ ]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# %pip install gymnasium
# %pip install 'stable-baselines3[extra]'

In [10]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
import gymnasium as gym
import os

# Colab Imports
if is_colab:
    %pip install gymnasium stable-baselines3[extra] pygame
    %pip install pyvirtualdisplay
    !apt-get install -y xvfb

    from pyvirtualdisplay import Display
    from IPython.display import Image
    import imageio

    display = Display(visible=0, size=(400, 300))
    display.start()


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Create enviroment for the model

In [15]:
enviroment_name = 'CartPole-v1'
env = gym.make(id=enviroment_name, render_mode="rgb_array")

## Test algorithm

In [16]:
episodes = 100
frames = []

for episode in range(1, episodes+1):
    state, _ = env.reset()
    done = False
    score = 0

    while not done:
        if is_colab:
            frames.append(env.render())
        action = env.action_space.sample()
        n_state, reward, done, info, _ = env.step(action=action)
        score += reward
    print(f'Episode:{episode} Score:{score}')
env.close()
if is_colab:
    imageio.mimsave('./random_agent.gif', frames, fps=30)
    Image('./random_agent.gif')

Episode:1 Score:17.0
Episode:2 Score:24.0
Episode:3 Score:35.0
Episode:4 Score:10.0
Episode:5 Score:21.0
Episode:6 Score:16.0
Episode:7 Score:13.0
Episode:8 Score:18.0
Episode:9 Score:17.0
Episode:10 Score:12.0
Episode:11 Score:17.0
Episode:12 Score:54.0
Episode:13 Score:12.0
Episode:14 Score:25.0
Episode:15 Score:12.0
Episode:16 Score:29.0
Episode:17 Score:28.0
Episode:18 Score:13.0
Episode:19 Score:14.0
Episode:20 Score:11.0
Episode:21 Score:24.0
Episode:22 Score:11.0
Episode:23 Score:16.0
Episode:24 Score:13.0
Episode:25 Score:20.0
Episode:26 Score:15.0
Episode:27 Score:14.0
Episode:28 Score:40.0
Episode:29 Score:48.0
Episode:30 Score:9.0
Episode:31 Score:17.0
Episode:32 Score:34.0
Episode:33 Score:38.0
Episode:34 Score:14.0
Episode:35 Score:25.0
Episode:36 Score:35.0
Episode:37 Score:24.0
Episode:38 Score:14.0
Episode:39 Score:13.0
Episode:40 Score:39.0
Episode:41 Score:25.0
Episode:42 Score:11.0
Episode:43 Score:49.0
Episode:44 Score:21.0
Episode:45 Score:18.0
Episode:46 Score:15.

/usr/local/lib/python3.12/dist-packages/imageio/plugins/pillow.py:410: DeprecationWarning: The keyword `fps` is no longer supported. Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Training the model

In [17]:
log_path = os.path.join(os.getcwd(), "Training", "Logs")
model_path = os.path.join(os.getcwd(), "Training", "Models")

In [18]:
env = gym.make(enviroment_name)  # no render_mode
env = DummyVecEnv([lambda: env])
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
model.learn(total_timesteps=20000)

Logging to /content/Training/Logs/PPO_1
-----------------------------
| time/              |      |
|    fps             | 1118 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 807          |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0077675143 |
|    clip_fraction        | 0.0898       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.687       |
|    explained_variance   | -0.0276      |
|    learning_rate        | 0.0003       |
|    loss                 | 8.71         |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.0137      |
|    value_loss           | 53           |
-------------------

## Save and Reload model

In [20]:
PPO_path = os.path.join(model_path, 'PPO_Model_CartPole')

In [21]:
model.save(PPO_path)

In [22]:
del model

In [23]:
model = PPO.load(PPO_path, env)

## Evaluation

In [24]:
evaluate_policy(model, env, n_eval_episodes=10)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(500.0), np.float64(0.0))

In [25]:
episodes = 500

if is_colab:
    eval_env = gym.make('CartPole-v1', render_mode='rgb_array')
    frames = []

    obs, _ = eval_env.reset()
    for _ in range(episodes):
        action, _ = model.predict(obs)
        obs, reward, done, _, info = eval_env.step(action)
        frames.append(eval_env.render())
        if done:
            obs, _ = eval_env.reset()

    eval_env.close()
    imageio.mimsave('trained_agent.gif', frames, fps=30)
    Image('trained_agent.gif')
else:
    eval_env = gym.make('CartPole-v1')
    obs, _ = eval_env.reset()
    for _ in range(episodes):
        action, _ = model.predict(obs)
        obs, reward, done, _, info = eval_env.step(action)
        if done:
            obs, _ = eval_env.reset()
    eval_env.close()

/usr/local/lib/python3.12/dist-packages/imageio/plugins/pillow.py:410: DeprecationWarning: The keyword `fps` is no longer supported. Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50).
  warnings.warn(
